# Stage 9 — Published D1 TENTACLES classifier discovery

Official source pin; train 10 authors’ classifiers on already verified D1 preprocessing. Separate from the four-way LOCO study.

## 1. Restore the EXACT already verified TENTACLES R environment

This notebook runs **the original authors’ 10 classifiers** on the already saved D1 `preProcess.obj`. It is the **published-D1 discovery track**, not a four-way LOCO benchmark. It never modifies the old `runs/tentacles_d1/` folders. It will stop if the R package revision, input RDS checksum, or optional dependencies do not match.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, shutil, subprocess, tarfile, tempfile, time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass
ROOT=Path('/content/drive/MyDrive/CMPS355_Research')
if not (ROOT/'configs/dataset.json').is_file():
    raise FileNotFoundError('Canonical CMPS355_Research missing from mounted MyDrive')
CONFIG=json.loads((ROOT/'configs/dataset.json').read_text())
EXPECTED_DATA_SHA='8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74'
SOURCE_SHA='069a26271e3885e015df25baa40a7a1140827d5f'
assert CONFIG['sha256'] == EXPECTED_DATA_SHA
LIB=Path('/content/cmps355_r_lib')
RSCRIPT=shutil.which('Rscript')
if RSCRIPT is None: raise RuntimeError('Rscript is not available on this Colab runtime')

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def installed_revision():
    env=os.environ.copy()
    env['CMPS355_R_LIB']=str(LIB)
    expr=('p<-Sys.getenv("CMPS355_R_LIB");'
          'if(dir.exists(p)).libPaths(c(p,.libPaths()));'
          'if(!requireNamespace("TENTACLES",quietly=TRUE))quit(status=2L);'
          'cat(tolower(utils::packageDescription("TENTACLES")[["RemoteSha"]]))')
    r=subprocess.run([RSCRIPT,'--vanilla','-e',expr],env=env,capture_output=True,text=True)
    return r.stdout.strip() if r.returncode==0 else None

rver=subprocess.check_output([RSCRIPT,'--vanilla','-e','cat(as.character(getRversion()))'],text=True).strip()
os_info=platform.freedesktop_os_release()
key=f"r{rver}_{os_info.get('ID','linux')}{os_info.get('VERSION_ID','unknown')}_{platform.machine()}"
cache=ROOT/'runs/environment_cache'/f'{key}.tar.gz'
manifest=ROOT/'runs/environment_cache'/f'{key}.json'
revision=installed_revision()
if revision!=SOURCE_SHA:
    if not cache.is_file() or not manifest.is_file():
        raise RuntimeError(f'Compatible pinned TENTACLES cache missing: {cache}. Run notebook 04 method verification, not a floating GitHub install.')
    meta=json.loads(manifest.read_text())
    expected={'r_version':rver,'os':os_info.get('ID','linux'),
              'os_version':os_info.get('VERSION_ID','unknown'),'arch':platform.machine()}
    if any(meta.get(k)!=v for k,v in expected.items()):
        raise RuntimeError('Cached package OS/R/architecture mismatch')
    if digest(cache)!=meta['sha256']: raise RuntimeError('Cached R library checksum mismatch')
    LIB.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        with tarfile.open(cache,'r:gz') as tf:
            for item in tf.getmembers():
                p=Path(item.name).parts
                if not p or p[0]!='r_library' or '..' in p:
                    raise RuntimeError('Unexpected package archive member')
            tf.extractall(tmp,filter='data')
        for package in (Path(tmp)/'r_library').iterdir():
            if package.is_dir() and not (LIB/package.name).exists():
                shutil.copytree(package,LIB/package.name)
    revision=installed_revision()
if revision!=SOURCE_SHA:
    raise RuntimeError(f'Official source pin does not match: {revision}')
ENV=os.environ.copy()
ENV['CMPS355_R_LIB']=str(LIB)
ENV.update({'OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1'})
print('Official TENTACLES SHA verified:',revision)
print('R version:',rver,'Cache key:',key,'Workspace:',ROOT)
try:
    import psutil
    print('Colab RAM available (GiB):',round(psutil.virtual_memory().available/1024**3,2))
except ImportError:
    pass


Mounted at /content/drive
Official TENTACLES SHA verified: 069a26271e3885e015df25baa40a7a1140827d5f
R version: 4.6.1 Cache key: r4.6.1_ubuntu24.04_x86_64 Workspace: /content/drive/MyDrive/CMPS355_Research
Colab RAM available (GiB): 11.44


## 2. Locate and verify completed D1 preprocessing

In [2]:
runs=ROOT/'runs/tentacles_d1'
valid=[]
for path in runs.iterdir():
    report=path/'report.json'
    if not path.is_dir() or not report.is_file(): continue
    m=json.loads(report.read_text())
    rds=path/'d1_preprocessed.rds'
    entry=m.get('files',{}).get('d1_preprocessed.rds')
    if (m.get('status')=='PASS_D1_PREPROCESS' and m.get('source_commit')==SOURCE_SHA
        and m.get('dataset_sha256')==EXPECTED_DATA_SHA and entry
        and rds.is_file() and digest(rds)==entry['sha256']):
        valid.append((path,rds,m))
if not valid: raise RuntimeError('No checksum-verified Stage 9 original D1 preprocessing found')
source,rds,original=sorted(valid,key=lambda x:x[0].name)[-1]
print('Found accepted D1 preProcess:',rds)
print('SHA-256:',digest(rds))


Found accepted D1 preProcess: /content/drive/MyDrive/CMPS355_Research/runs/tentacles_d1/20261107T072703872271Z/d1_preprocessed.rds
SHA-256: b04bc485b667746828b2e67450f9f698a84f938e2960f395c1aaed3a5d02e491


## 3. API/dependency preflight: fail closed BEFORE expensive training

In [3]:
def run_and_log(command,log_path,env=ENV):
    log_path=Path(log_path)
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,env=env,bufsize=1) as proc:
            for line in proc.stdout:
                log.write(line)
                print(line,end='',flush=True)
            return proc.wait()

def write_report(output,stage,success,extra):
    output=Path(output)
    files={}
    for p in sorted(output.rglob('*')):
        if p.is_file() and p.name!='report.json':
            files[str(p.relative_to(output))]={'sha256':digest(p),'bytes':p.stat().st_size}
    report=dict(status='PASS_CLASSIFIERS_ONLY' if success else 'FAIL_CLASSIFIERS',
                stage=stage,source_commit=SOURCE_SHA,dataset_sha256=EXPECTED_DATA_SHA,
                scope='official R fit only; no external heldout predictions or consensus yet',
                created_utc=datetime.now(timezone.utc).isoformat(),files=files,**extra)
    tmp=output/'report.json.tmp'
    tmp.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
    os.replace(tmp,output/'report.json')
    return report
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
preflight=ROOT/'runs/tentacles_stage9_preflight'/stamp
preflight.mkdir(parents=True,exist_ok=False)
probe_script=ROOT/'r/tentacles_stage9_preflight.R'
assert probe_script.is_file()
rc=run_and_log([RSCRIPT,'--vanilla',str(probe_script),str(rds),str(preflight),SOURCE_SHA],preflight/'preflight.log')
print('API documents saved in:',preflight)
if rc!=0 or 'STAGE9_PREFLIGHT_PASS' not in (preflight/'preflight.log').read_text():
    raise RuntimeError('Official API/dependency gate failed; no 10-model run was launched')


TENTACLES_SHA=069a26271e3885e015df25baa40a7a1140827d5f
TENTACLES_VERSION=0.1.0
R_VERSION=4.6.1
D1_PREPROCESSED_CLASS=preProcess.obj
OPTIONAL_DEPENDENCIES_MISSING=tidymodels,bonsai,lightgbm,earth,xgboost,mixOmics,plsmod
Error: Preflight stopped: restore/install missing optional packages before a 10-model run; see optional_dependency_check.csv
Execution halted
API documents saved in: /content/drive/MyDrive/CMPS355_Research/runs/tentacles_stage9_preflight/20261107T185619073385Z


RuntimeError: Official API/dependency gate failed; no 10-model run was launched

## 4. Official 10-model discovery fit (the expensive step)

The ten model-selector pairs and 5-fold, five-candidate `f_meas` tuning configuration come from the 2026 paper. The stage stores the full native object, component structures, configuration, warnings and provenance. It does NOT claim to reproduce the published 28-gene signature until `getConsensus` is separately verified.

In [ ]:
run=ROOT/'runs/tentacles_paper_d1_classifiers'/stamp
run.mkdir(parents=True,exist_ok=False)
script=ROOT/'r/tentacles_stage9_paper_d1_classifiers.R'
if not script.is_file(): raise FileNotFoundError(script)
import time
start=time.monotonic()
rc=run_and_log([RSCRIPT,'--vanilla',str(script),str(rds),str(run),SOURCE_SHA],run/'classifiers.log')
log=(run/'classifiers.log').read_text(errors='replace')
pass_status=rc==0 and 'CLASSIFIERS_PASS=TRUE' in log and (run/'runclassifiers_finished.rds').is_file()
summary=write_report(run,'published_D1_classifiers',pass_status,{
 'exit_code':rc,'elapsed_wall_seconds':round(time.monotonic()-start,1),
 'preprocess_source':str(rds.relative_to(ROOT)),
 'preprocess_sha256':digest(rds),
 'runner_sha256':digest(script),
 'model_count_requested':10,
 'next_step':'Inspect getConsensus API + native importance, then generate actual consensus and validate source fidelity'
})
print('Saved output:',run,'Final stage status:',summary['status'])
if not pass_status: raise RuntimeError('Official D1 classifier stage failed; inspect saved classifiers.log')
